#  Comprehensive Solar Energetic Electron event Catalogue (CoSEE-Cat), Data Release 1
This notebook is set up to analyse the CoSEE-Cat DR1 dataset (see Warmuth 2025).

This notebook reads in the CoSEE-cat .csv, and finds the solar energetic electron events that are not associated with a STIX X-ray flare. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

path = Path(r"C:\Users\pauls\SEPs\coseecat_dr1\data\CoSEE_full.csv")
print(path.exists(), f"{path.stat().st_size / 1e6:.2f} MB")

True 0.13 MB


In [3]:
# look at the raw text file first

with open(path, "r", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        print(i, line[:300].rstrip())
        if i >= 15:
            break
        

0 event_id,event_date,solo_dist,solo_lon,solo_lat,epd_tonset,epd_delta_t,epd_tpeak,epd_ipeak,epd_epeak,tsa_itime,vda_itime,vda_time_unc,vda_lpath,vda_path_unc,nominal_path,epd_aniso,epd_compo,epd_series,epd_dispersive,goes_class,goes_estim,stix_status,stix_tpeak,stix_emax,stix_tpeak_epd,stix_tpeak_rpw
1 2011170942,2020-11-17,0.928,122.2,-6.0,2020-11-17T09:42:30,60,2020-11-17T10:21:30,3.42E+04,43,2020-11-17T09:27:27,,,,,1.06,large,impulsive,Y,,,C1,1,2020-11-17T09:32:45,25.0,2020-11-17T09:26:49,,929.0,-285,-18.1,-167.1,340.6,12786,2,1,969.0,-317.0,-18.8,-156.3,351.4,12786,flare,510.0,628.0,32.4,157.8
2 2011171841,2020-11-17,0.926,122.1,-6.0,2020-11-17T18:41:30,60,2020-11-17T18:48:30,2.70E+04,43,2020-11-17T18:26:29,2020-11-17T18:25:24,0.9,1.15,0.03,1.06,large,impulsive,Y,Y,,B6,1,2020-11-17T18:26:57,15.0,2020-11-17T18:26:57,2020-11-17T18:26:57,949.0,-383,-22.6,-154.9,347.9,12786,1,1,983.0,-267.0,-16
3 2011181056,2020-11-18,0.924,122.1,-6.0,2020-11-18T10:56:30,60,,,24,2020-11-18T10:34:33,,,

In [4]:
# Read in the file

df = pd.read_csv(path)
# If Cell 2 shows comment lines, use:  pd.read_csv(path, comment="#")
# If it shows fill values such as -9999: pd.read_csv(path, na_values=[-9999, "-9999"])

print(df.shape)
df.head()

(303, 99)


,event_id,event_date,solo_dist,solo_lon,solo_lat,epd_tonset,epd_delta_t,epd_tpeak,epd_ipeak,epd_epeak,...,ss_fr_delay,sw_condition,mag_polarity,ip_complex,ip_confidence,conn_carr_lon,conn_lon_unc,conn_hgs_lat,conn_lat_unc,conn_conf
0,2011170942,2020-11-17,0.928,122.2,-6.0,2020-11-17T09:42:30,60,2020-11-17T10:21:30,34200.0,43.0,...,30.0,NaN,NaN,n,2,333.28,4.00,-30.77,5.75,3
1,2011171841,2020-11-17,0.926,122.1,-6.0,2020-11-17T18:41:30,60,2020-11-17T18:48:30,27000.0,43.0,...,20.0,NaN,NaN,n,2,332.29,6.50,-31.22,9.25,3
2,2011181056,2020-11-18,0.924,122.1,-6.0,2020-11-18T10:56:30,60,NaN,NaN,24.0,...,4.0,NaN,NaN,y,2,329.34,3.16,-31.16,3.41,3
3,2011181214,2020-11-18,0.924,122.1,-6.0,2020-11-18T12:14:30,60,2020-11-18T12:15:30,2900.0,43.0,...,3.0,NaN,NaN,y,2,329.34,3.16,-31.16,3.41,3
4,2011181331,2020-11-18,0.924,122.1,-6.0,2020-11-18T13:31:30,60,2020-11-18T13:41:30,7640.0,43.0,...,2.0,NaN,NaN,y,2,329.34,3.16,-31.16,3.41,3


In [6]:
# Structure overview
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 99 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   event_id           303 non-null    int64  
 1   event_date         303 non-null    str    
 2   solo_dist          303 non-null    float64
 3   solo_lon           303 non-null    float64
 4   solo_lat           303 non-null    float64
 5   epd_tonset         303 non-null    str    
 6   epd_delta_t        303 non-null    int64  
 7   epd_tpeak          300 non-null    str    
 8   epd_ipeak          300 non-null    float64
 9   epd_epeak          303 non-null    float64
 10  tsa_itime          303 non-null    str    
 11  vda_itime          126 non-null    str    
 12  vda_time_unc       126 non-null    float64
 13  vda_lpath          126 non-null    float64
 14  vda_path_unc       126 non-null    float64
 15  nominal_path       303 non-null    float64
 16  epd_aniso          226 non-null    st

In [7]:
# Columns and missing data

summary = pd.DataFrame({
    "dtype": df.dtypes,
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(1),
    "n_unique": df.nunique(),
})
summary

,dtype,n_missing,pct_missing,n_unique
event_id,int64,0,0.0,303
event_date,str,0,0.0,149
solo_dist,float64,0,0.0,210
solo_lon,float64,0,0.0,197
solo_lat,float64,0,0.0,97
...,...,...,...,...
conn_carr_lon,float64,0,0.0,206
conn_lon_unc,float64,0,0.0,116
conn_hgs_lat,float64,0,0.0,199
conn_lat_unc,float64,0,0.0,129


,count,mean,std,min,25%,50%,75%,max
event_id,303.0,2.175977e+09,5.864500e+07,2.011171e+09,2.112311e+09,2.205102e+09,2.210292e+09,2.212302e+09
solo_dist,303.0,7.006898e-01,2.097065e-01,3.130000e-01,4.980000e-01,7.090000e-01,9.095000e-01,1.015000e+00
solo_lon,303.0,2.718350e+01,9.568474e+01,-1.613000e+02,-2.695000e+01,-1.650000e+01,1.363000e+02,1.792000e+02
solo_lat,303.0,1.407921e+00,4.747694e+00,-7.800000e+00,-2.550000e+00,1.900000e+00,5.050000e+00,8.000000e+00
epd_delta_t,303.0,7.782178e+01,7.461296e+01,1.500000e+01,6.000000e+01,6.000000e+01,6.000000e+01,6.000000e+02
...,...,...,...,...,...,...,...,...
conn_carr_lon,303.0,1.701106e+02,1.144583e+02,9.000000e-01,7.366500e+01,1.375000e+02,2.862300e+02,3.577500e+02
conn_lon_unc,303.0,3.350099e+00,3.390812e+00,0.000000e+00,6.500000e-01,1.890000e+00,5.190000e+00,1.972000e+01
conn_hgs_lat,303.0,5.822145e+00,2.301041e+01,-5.076000e+01,-1.685000e+01,1.587000e+01,2.195000e+01,6.613000e+01
conn_lat_unc,303.0,2.691353e+00,2.237363e+00,0.000000e+00,9.800000e-01,2.080000e+00,4.130000e+00,1.167000e+01


In [10]:
# check stix_status

df["stix_status"].value_counts().sort_index()

stix_status
0     15
1    288
Name: count, dtype: int64

In [11]:
def stix_category(r):
    if pd.isna(r["stix_tpeak"]):
        return "no flare time (a or b)"
    if pd.isna(r["stix_hgs_lat"]):
        return "c) flare, no position"
    return "flare + position"

df["stix_cat"] = df.apply(stix_category, axis=1)
pd.crosstab(df["stix_cat"], df["stix_status"], margins=True)

stix_status,0,1,All
stix_cat,,,
"c) flare, no position",0,63,63
flare + position,0,205,205
no flare time (a or b),15,20,35
All,15,288,303


In [12]:
cols = ["event_id", "event_date", "stix_status", "stix_tpeak", "stix_emax",
        "goes_estim", "stix_epd_qual", "eui_status", "rpw_status"]
df.loc[df["stix_cat"] == "no flare time (a or b)", cols].sort_values("stix_status")

,event_id,event_date,stix_status,stix_tpeak,stix_emax,goes_estim,stix_epd_qual,eui_status,rpw_status
8,2011240701,2020-11-24,0,NaN,NaN,NaN,3,0,1
9,2011241331,2020-11-24,0,NaN,NaN,NaN,3,0,1
10,2011241735,2020-11-24,0,NaN,NaN,NaN,3,0,1
11,2011291337,2020-11-29,0,NaN,NaN,NaN,3,0,1
13,2012100809,2020-12-10,0,NaN,NaN,NaN,3,0,1
12,2012100748,2020-12-10,0,NaN,NaN,NaN,3,0,1
14,2012101344,2020-12-10,0,NaN,NaN,NaN,3,0,1
15,2012102049,2020-12-10,0,NaN,NaN,NaN,3,0,1
19,2012130951,2020-12-13,0,NaN,NaN,NaN,3,0,1
16,2012102346,2020-12-10,0,NaN,NaN,NaN,3,0,1


In [13]:
cols = ["event_id", "event_date", "stix_status", "stix_epd_qual",
        "stix_tpeak_epd", "stix_emax", "goes_estim", "eui_status", "rpw_status"]
no_flare = df[df["stix_cat"] == "no flare time (a or b)"]
no_flare[cols].sort_values(["stix_status", "event_date"])

,event_id,event_date,stix_status,stix_epd_qual,stix_tpeak_epd,stix_emax,goes_estim,eui_status,rpw_status
8,2011240701,2020-11-24,0,3,NaN,NaN,NaN,0,1
9,2011241331,2020-11-24,0,3,NaN,NaN,NaN,0,1
10,2011241735,2020-11-24,0,3,NaN,NaN,NaN,0,1
11,2011291337,2020-11-29,0,3,NaN,NaN,NaN,0,1
12,2012100748,2020-12-10,0,3,NaN,NaN,NaN,0,1
13,2012100809,2020-12-10,0,3,NaN,NaN,NaN,0,1
14,2012101344,2020-12-10,0,3,NaN,NaN,NaN,0,1
15,2012102049,2020-12-10,0,3,NaN,NaN,NaN,0,1
16,2012102346,2020-12-10,0,3,NaN,NaN,NaN,0,1
17,2012111055,2020-12-11,0,3,NaN,NaN,NaN,0,1


In [14]:
pd.crosstab(no_flare["stix_status"], no_flare["stix_epd_qual"])

stix_epd_qual,3
stix_status,
0,15
1,20


In [19]:
for status, dates in no_flare.groupby("stix_status")["event_date"]:
    print(f"\nstix_status = {status} ({len(dates)} events)")
    print(sorted(dates.tolist()))


stix_status = 0 (15 events)
['2020-11-24', '2020-11-24', '2020-11-24', '2020-11-29', '2020-12-10', '2020-12-10', '2020-12-10', '2020-12-10', '2020-12-10', '2020-12-11', '2020-12-11', '2020-12-13', '2021-04-19', '2022-04-28', '2022-04-28']

stix_status = 1 (20 events)
['2020-11-20', '2021-02-14', '2021-05-24', '2021-05-24', '2021-05-24', '2021-05-28', '2021-07-16', '2021-07-18', '2021-12-04', '2021-12-05', '2022-03-18', '2022-03-19', '2022-05-03', '2022-10-18', '2022-10-29', '2022-10-29', '2022-10-29', '2022-10-30', '2022-12-24', '2022-12-24']


In [16]:
pd.crosstab(df["stix_cat"], df["stix_epd_qual"], margins=True)

stix_epd_qual,1,2,3,All
stix_cat,,,,
"c) flare, no position",43,16,4,63
flare + position,96,104,5,205
no flare time (a or b),0,0,35,35
All,139,120,44,303


In [20]:
# Catalague also uses EUI for flare location. 

needs = df[df["stix_hgs_lat"].isna()].copy()
print(len(needs))

needs["has_eui"] = needs["eui_hgs_lat1"].notna()
pd.crosstab(needs["stix_cat"], needs["has_eui"], margins=True)

98


has_eui,False,True,All
stix_cat,,,
"c) flare, no position",21,42,63
no flare time (a or b),22,13,35
All,43,55,98


In [22]:
# Do STIX and EUI agree where both exist?

both = df[df["stix_hgs_lat"].notna() & df["eui_hgs_lat1"].notna()]
print(len(both))
print((both["eui_hgs_lon1"] - both["stix_hgs_lon"]).describe())
print((both["eui_hgs_lat1"] - both["stix_hgs_lat"]).describe())

130
count    130.000000
mean       1.246923
std       34.228120
min     -103.400000
25%       -2.200000
50%       -0.200000
75%        0.975000
max      351.200000
dtype: float64
count    130.000000
mean       0.650000
std        9.127021
min      -45.700000
25%       -0.600000
50%        0.400000
75%        1.700000
max       49.700000
dtype: float64


In [23]:
df["eui_type1"].value_counts(dropna=False)

eui_type1
NaN                                                  109
flare                                                 88
flare, jet                                            59
flare, erupting filament                              11
flare, loop opening                                    9
flare, fan-like eruption                               9
erupting filament                                      2
flare, loop-like eruption                              2
flare, loop-like eruption, fan-like eruption           2
flare, jet, loop opening                               2
flare,  loop opening                                   1
loop opening                                           1
jet                                                    1
flare, erupting filament, loop-like eruption           1
flare, loop-like eruption                              1
flare, loop opening, fan-like eruption                 1
flare, loop-opening                                    1
flare, loop-like     